In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# Claims Processing Pipeline
# ========================================
 
from pyspark.sql.functions import *
from delta.tables import DeltaTable
 
try:
    # Read Claims Data
    claims_df = spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(f"{source_path}/claims")
 
    # Add ingestion timestamp
    claims_df = claims_df.withColumn(
        "ingestion_time",
        current_timestamp()
    )
 
    # Write to Bronze Layer
    claims_df.write.format("delta") \
        .mode("append") \
        .save(f"{bronze_path}/claims")
 
    # Read Bronze Data
    bronze_claims_df = spark.read.format("delta") \
        .load(f"{bronze_path}/claims")
 
    # Data Cleaning
    clean_claims_df = bronze_claims_df.filter(
        col("claim_amount") > 0
    )
 
    clean_claims_df = clean_claims_df.withColumn(
        "claim_status",
        upper(trim(col("claim_status")))
    )
 
    # Write to Silver Layer
    clean_claims_df.write.format("delta") \
        .mode("overwrite") \
        .save(f"{silver_path}/claims_clean")
 
    # Gold Layer Analytics
    analytics_df = clean_claims_df.groupBy(
        "claim_status"
    ).agg(
        sum("claim_amount").alias("total_claim_amount"),
        count("claim_id").alias("total_claims")
    )
 
    analytics_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true").save(f"{gold_path}/claims_analytics")
    display(analytics_df)    
 
    print("Claims Processing Pipeline Completed Successfully")
 
except Exception as e:
    print(f"Error occurred: {str(e)}")
 